In [ ]:
import os
import numpy as np
import pandas as pd
import arviz as az
import pymc as pm
import matplotlib.pyplot as plt
from scipy.special import expit as logistic
from scipy.stats import binom
from scipy.stats import multinomial
from scipy.special import softmax
import pytensor.tensor as tt

# Binomial, Poisson, Multinomial regression

<img style='float: right; margin-right: 100px' src="figs/abacus.jpg" width="40%">


## Agenda

* Binomial regression
    * Social chimpanzees
    * UC Berkeley admission bias

<br>


* Poisson regression
    * Tool counts and population size

<br>


* Multinomial regression
    * Career choice based on expected income




## Binomial regression

* We have seen it in previous lectures, e.g., globe tossing example and in the MCMC lecture.

* It is s appropriate for count data with a fixed maximum

    * Fixed number of globe tosses
    
    * Fixed number of coin tosses
    
    




### Chimpanzee experiment

<img src="figs/chimpanzee-experiment.png" width="45%"/>

* We have a look at the chimpanzee data

In [ ]:
d = pd.read_csv(os.path.join("Data", "chimpanzees.csv"), delimiter=";")
d

* We consider the predictors
    * `condition` whether the partner was on the other side of the table
    * `prosoc_left` whether the prosocial option was on the left

<br>

* And the outcome variable
    * `pulled_left` whether the chimpanzee pulled the left lever

<br>

In [ ]:
d.actor -= 1 # start actor indexes on 0

In [ ]:
d['treatment'] = d.prosoc_left + 2*d.condition # we index starting from 0 (that's why there is no "1 +" at the beginning)

In [ ]:
# just plotting the first record for each combination
d.groupby('treatment').first()[['condition','prosoc_left']]

#### Model

\begin{align*}
L_i  \sim & \; \mathrm{Binomial}(1,p_i) \\
\mathrm{logit}(p_i) = & \; \alpha_{\mathrm{ACTOR}[i]} + \beta_{\mathrm{TREATMENT}[i]} \\
\alpha_j  \sim & \; \text{To be determined} \\
\beta_k  \sim & \; \text{To be determined} \\
&\\
\end{align*}

<br>

* Remember that $\mathrm{logit}(p_i) = \alpha_{\mathrm{ACTOR}[i]} + \beta_{\mathrm{TREATMENT}[i]}$ is equivalent to $p_i = \mathrm{invlogit}(\alpha_{\mathrm{ACTOR}[i]} + \beta_{\mathrm{TREATMENT}[i]})$ where $\mathrm{invlogit}$ is the _inverse_ logit function (a.k.a. *logistic* function).     
    * We will use this formulation in PyMC models

<br>

* Due to the logit link function, $\alpha_{\mathrm{ACTOR}[i]} + \beta_{\mathrm{TREATMENT}[i]}$ is in logit or log-odds scale
    * This has an impact on the meaning of the parameters
    * The meaning is not a probability value as in previous instances of Binomial regression

<br>

* Note $\mathrm{Binomial}$ with $n=1$ is equivalent to $Bernoulli$
    
<br>

#### Finding prior for $\alpha$

* First we speculatively explore possible priors for $\alpha$

* To this end, we consider a simpler version of the model without $\beta$ or categories

\begin{align*}
L_i  \sim & \; \mathrm{Binomial}(1,p_i) \\
\mathrm{logit}(p_i) = & \; \alpha \\
\alpha  \sim & \; \mathrm{Normal}(0,\omega)
\end{align*}

* We consider different values for $\omega$.

In [ ]:
# ω = 10
with pm.Model() as m11_1:
    α = pm.Normal('α',mu=0,sigma=10)
    p = pm.Deterministic('p',pm.math.invlogit(α))
    L = pm.Binomial('L',p=p,n=1,observed=d.pulled_left)

# ω = 1.5
with pm.Model() as m11_1_a:
    α = pm.Normal('α',mu=0,sigma=1.5) # found experimentally...
    p = pm.Deterministic('p',pm.math.invlogit(α))
    L = pm.Binomial('L',p=p,n=1,observed=d.pulled_left)

In [ ]:
m11_1

In [ ]:
pm.model_to_graphviz(m11_1)

#### Prior predictive simulation

* We will see that selecting priors is not as easy as for models without link functions

In [ ]:
prior_trace_11_1   = pm.sample_prior_predictive(model=m11_1).prior
prior_trace_11_1_a = pm.sample_prior_predictive(model=m11_1_a).prior

In [ ]:
az.plot_density(
    [prior_trace_11_1['p'], prior_trace_11_1_a['p']],
    data_labels=['α ~ Normal(0,10)','α ~ Normal(0,1.5)'],
    colors=['black','blue'],
    point_estimate=None)
plt.title('Prior predictive simulation')
plt.xlabel('Prior prob pull left, i.e., p')
plt.ylabel('Density');

* When $\sigma = 10$ the prior "thinks" that the chimpanzee either always or never pulls left
* With $\sigma = 1.5$ the prior becomes "flat", i.e., over all values of the ($\log$) parameter space




* Now we try to find a good prior for the $\beta$ parameter, so we extend the model to include it

In [ ]:
with pm.Model() as m11_2:
    α = pm.Normal('α',mu=0,sigma=1.5)
    β = pm.Normal('β',mu=0,sigma=10,shape=d.treatment.unique().size)
    p = pm.Deterministic('p',pm.math.invlogit(α + β[d.treatment]))
    L = pm.Binomial('L',p=p,n=1,observed=d.pulled_left)
    
with pm.Model() as m11_3:
    α = pm.Normal('α',mu=0,sigma=1.5)
    β = pm.Normal('β',mu=0,sigma=0.5,shape=d.treatment.unique().size)
    p = pm.Deterministic('p',pm.math.invlogit(α + β[d.treatment]))
    L = pm.Binomial('L',p=p,n=1,observed=d.pulled_left)

In [ ]:
pm.model_to_graphviz(m11_2)

In [ ]:
prior_trace_11_2 = pm.sample_prior_predictive(model=m11_2).prior
prior_trace_11_3 = pm.sample_prior_predictive(model=m11_3).prior

* Since we are interested in the difference between treatments, we compare the difference in probability of the first two treatments
    * To this end we compute the contrast for these prior predictive distributions
    * We only compare the first two because it is a prior predictive check
        * Before adding the data all 4 treatment parameters produce the distribution

<br>

* Note the use of `logistic` in the cell below
    * The `logistic` function is the inverse of the logit
    * We must apply the entire linear model, i.e., $\mathit{logistic}(\alpha + \beta)$
    * It is used to interpret the prior predictive check in the probability scale

<br>

In [ ]:
prior_m = lambda trace, treatment : logistic(trace.sel(β_dim_0=treatment)['α'] + trace.sel(β_dim_0=treatment)['β'])
diff_m_11_2 = np.abs(prior_m(prior_trace_11_2,0) - prior_m(prior_trace_11_2,1))
diff_m_11_3 = np.abs(prior_m(prior_trace_11_3,0) - prior_m(prior_trace_11_3,1))

In [ ]:
az.plot_density(
    [diff_m_11_2, diff_m_11_3],
    data_labels=['$β_k$ ~ Normal(0,10)','$β_k$ ~ Normal(0,0.5)'],
    colors=['black','blue'],
    point_estimate=None)
plt.title('Prior predictive simulation')
plt.xlabel('Prior diff between treatments 0 and 1')
plt.ylabel('Density');

In [ ]:
diff_m_11_3.mean()

* As before, the flat prior ($\omega = 1.5$) gives high probability to extreme outcomes; either treatments produce completely different results or the same (differences 0 or 1)

<br>

* With $\omega=0.5$ all density is concreted in a small absolute distance between outcomes
    * This is a more realistic possibility in this type of experiment

<br>

* Now we move on to sample the posterior
    * Note the `pm.Data` lines; these are only needed to sample with custom values for predictors

In [ ]:
actor_idx, actors = pd.factorize(d.actor)
treat_idx, treatments = pd.factorize(d.treatment)

In [ ]:
with pm.Model() as m11_4:
    α = pm.Normal('α',mu=0,sigma=1.5,shape=actors.size)
    β = pm.Normal('β',mu=0,sigma=0.5,shape=treatments.size)
    
    # The following two lines allow for setting specific values for the predictors
    actor_ids = pm.Data('actor_ids',actor_idx)
    treatment_ids = pm.Data('treatment_ids',treat_idx)
    p = pm.Deterministic('p',pm.math.invlogit(α[actor_ids] + β[treatment_ids]))
    L = pm.Binomial('L',p=p,n=1,observed=d.pulled_left)

In [ ]:
pm.model_to_graphviz(model=m11_4)

In [ ]:
trace_m_11_4 = pm.sample(model=m11_4)

In [ ]:
pm.summary(trace_m_11_4,var_names=['α','β'],round_to=2)

* These values are better discussed using a forest plot

<br>

* Below we plot the values for the actor intercept in probability scale
    * Note the use of `transform=logistic`


In [ ]:
pm.plot_forest(trace_m_11_4, var_names=['α'], transform=logistic, hdi_prob=.95, combined=True);

* Left means preference for right lever (not left)

* The data above is not very useful in understanding the prosocial behaviour of the chimpanzees

* Below we plot the $\beta$ intercepts to better understand the results for each treatment

    * The plots show the distributions in the logit scale; a.k.a. log-odds scale

In [ ]:
ax = pm.plot_forest(trace_m_11_4, var_names=['β'], hdi_prob=.95, combined=True)
ax[0].set_yticklabels(["L/P", "R/P", "L/N", "R/N"]);

* What we are actually interested in is whether chimpanzees take the prosocial option more often when there is a partner.

* Below we compute the *contrast* that is the difference between the options above

* **[QUESTION]: What is the scale of the contrast distributions below? Could we apply some transformation to improve interpretability?**

<!-- * Note that the scale is log-odds!

    * Recall: the log-odds of an event is just the log of the probability the event happens divided by the probability it does not happen. 

    * Before plotting the contrasts, we plot the logit function to get some intuition  -->

In [ ]:
posterior_diff = lambda trace, treatment : trace.posterior['β'].sel(β_dim_0=treatment)
diff_02 = posterior_diff(trace_m_11_4,0) - posterior_diff(trace_m_11_4,2)
diff_13 = posterior_diff(trace_m_11_4,1) - posterior_diff(trace_m_11_4,3)
pm.plot_forest([diff_02,diff_13],model_names=['diff_02','diff_13'], hdi_prob=.95, combined=True);

* For `diff_02` positive values indicate a prosocial behaviour
    * Positive values in this plot are caused by low probability of pulling left (i.e., high probability of pulling right) in the prosocial option.

<br>

* Conversely, for `diff_13`, lower negative values indicate the prosocial choice
    * As before, negative values are caused by high probability of pulling left on the prosocial option

<br>

* Overall, based on the results above, we can conclude that chimpanzees did not show a strong preference for the prosocial choice 

#### Posterior predictive check

* First we compute the (empirical) mean proportion of left lever pull for each treatment and actor

In [ ]:
pl = d.groupby(['actor','treatment']).mean().pulled_left.unstack()
pl

* To perform the posterior predictive check we sample from our posterior and (visually) check whether the results match

In [ ]:
with m11_4:
    pm.set_data({"actor_ids": np.repeat(range(7), 4), "treatment_ids": list(range(4)) * 7})
    p_post = pm.sample_posterior_predictive(trace_m_11_4, var_names=["p"]).posterior_predictive["p"]
p_mu = np.array(p_post.mean(["chain", "draw"])).reshape((7, 4))

In [ ]:
# copied from (https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb)

pl_val = pl.stack().values
_, (ax0, ax1) = plt.subplots(2, 1, figsize=(12, 6))
alpha, xoff, yoff = 0.6, 0.3, 0.05

ax0.plot([7 * 4 - 0.5] * 2, [0, 1], c="k", alpha=0.4, lw=1)
ax0.axhline(0.5, ls="--", c="k", alpha=0.4)
for actor in range(len(actors)):
    ax0.plot(
        [actor * 4, actor * 4 + 2],
        [pl.loc[actor, 0], pl.loc[actor, 2]],
        "-",
        c="b",
        alpha=alpha,
    )
    ax0.plot(
        [actor * 4 + 1, actor * 4 + 3],
        [pl.loc[actor, 1], pl.loc[actor, 3]],
        "-",
        c="b",
        alpha=alpha,
    )
    ax0.plot(
        [actor * 4, actor * 4 + 1],
        [pl.loc[actor, 0], pl.loc[actor, 1]],
        "o",
        c="b",
        fillstyle="none",
        ms=6,
        alpha=alpha,
    )
    ax0.plot(
        [actor * 4 + 2, actor * 4 + 3],
        [pl.loc[actor, 2], pl.loc[actor, 3]],
        "o",
        c="b",
        ms=6,
        alpha=alpha,
    )
    ax0.plot([actor * 4 - 0.5] * 2, [0, 1], c="k", alpha=0.4, lw=1)
    ax0.text(actor * 4 + 0.5, 1.1, f"actor {actor + 1}", fontsize=12)
    if actor == 0:
        ax0.text(actor * 4 - xoff, pl.loc[actor, 0] - 2 * yoff, "R/N")
        ax0.text(actor * 4 + 1 - xoff, pl.loc[actor, 1] + yoff, "L/N")
        ax0.text(actor * 4 + 2 - xoff, pl.loc[actor, 2] - 2 * yoff, "R/P")
        ax0.text(actor * 4 + 3 - xoff, pl.loc[actor, 3] + yoff, "L/P")
ax0.set_xticks([])
ax0.set_ylabel("proportion left lever", labelpad=10)
ax0.set_title("observed proportions", pad=25)

ax1.plot([range(28), range(28)], az.hdi(p_post)['p'].T, "k-", lw=2, alpha=alpha)
ax1.plot([7 * 4 - 0.5] * 2, [0, 1], c="k", alpha=0.4, lw=1)
ax1.axhline(0.5, ls="--", c="k", alpha=0.4)
for actor in range(len(actors)):
    ax1.plot(
        [actor * 4, actor * 4 + 2],
        [p_mu[actor, 0], p_mu[actor, 2]],
        "-",
        c="k",
        alpha=alpha,
    )
    ax1.plot(
        [actor * 4 + 1, actor * 4 + 3],
        [p_mu[actor, 1], p_mu[actor, 3]],
        "-",
        c="k",
        alpha=alpha,
    )
    ax1.plot(
        [actor * 4, actor * 4 + 1],
        [p_mu[actor, 0], p_mu[actor, 1]],
        "o",
        c="k",
        fillstyle="none",
        ms=6,
        alpha=alpha,
    )
    ax1.plot(
        [actor * 4 + 2, actor * 4 + 3],
        [p_mu[actor, 2], p_mu[actor, 3]],
        "o",
        c="k",
        ms=6,
        alpha=alpha,
    )
    ax1.plot([actor * 4 - 0.5] * 2, [0, 1], c="k", alpha=0.4, lw=1)
    ax1.text(actor * 4 + 0.5, 1.1, f"actor {actor + 1}", fontsize=12)
ax1.set_xticks([])
ax1.set_ylabel("proportion left lever", labelpad=10)
ax1.set_title("posterior predictions", pad=25)
plt.tight_layout();

* Our model expects no changes between having or not a partner

* Variation in results is mostly driven by the actor (i.e., the intercept $\alpha_j$)

* It seems that different actors respond better to treatments than others
    * We will come back to this when we discuss multi-level models

#### Split predictors for `condition` and `prosoc_left`

* Note that `condition` and `prosoc_left` are interactive parameters



* For the sake of the questions above, we rewrite the model without interactive parameters and compare them

In [ ]:
d

In [ ]:
with pm.Model() as m11_5:
    βcond = pm.Normal('βcond',mu=0,sigma=0.5,shape=d.condition.unique().size)
    βside = pm.Normal('βside',mu=0,sigma=0.5,shape=d.prosoc_left.unique().size)
    α = pm.Normal('α',mu=0,sigma=1.5,shape=d.actor.unique().size)
    p = pm.Deterministic('p',pm.math.invlogit(α[d.actor] + βside[d.prosoc_left] + βcond[d.condition]))
    T = pm.Binomial('T',n=1,p=p,observed=d.pulled_left)

In [ ]:
with m11_4:
    # restore original data
    pm.set_data({'actor_ids': d.actor, 'treatment_ids': d.treatment})
    trace_m_11_4 = pm.sample(idata_kwargs = { 'log_likelihood': True })
trace_m_11_5 = pm.sample(model=m11_5,idata_kwargs = { 'log_likelihood': True })

* `{ 'log_likelihood': True }` adds to the inference data object the log-likelihood for each observation in the model for each posterior sample. This is necessary for computing information criteria.

In [ ]:
trace_m_11_4.log_likelihood

In [ ]:
pm.compare({'m11_4': trace_m_11_4, 'm11_5': trace_m_11_5})

* The results are virtually identical, meaning that they have very similar expected predictive accuracy

### Binomial aggregated regression

<img style='float: right; margin-right: 100px' src="figs/college_admission.jpg" width="40%">

* In the previous model, we had one experiment outcome per record in the dataset
    * This is why we always set $n=1$ in the Binomial likelihood        
    * In the previous scenario, chimpanzees repeated the experiment exactly 18 times
    * Data could have been aggregated per chimpanzee
        * Have 7 records, 1 per chimpanzee, stating how many times they pulled left
<br>

* Now we study an example of this kind of Binomial regression
    * But the number of experiments per subject varies

<br>

* We will study a dataset of **admissions to UC Berkeley**

<br>

* We will try to determine whether there is gender bias in admissions

<!-- <img src="figs/college_admission.jpg" width="40%"> -->

In [ ]:
d_ucb = pd.read_csv(os.path.join("Data", "UCBadmit.csv"), delimiter=";")
d_ucb

#### Model 1

* First we consider a model with a single predictor for gender (across all departments)

\begin{align*}
A_i \sim & \; \mathrm{Binomial}(N_i, p_i)\\
\mathrm{logit}(p_i) = & \; \alpha_{\mathrm{GID}[i]}\\
\alpha_j \sim & \; \mathrm{Normal}(0,1.5)
\end{align*}

In [ ]:
gid = (d_ucb['applicant.gender'] == "female").astype(int).values # 0 male, 1 female
with pm.Model() as m11_7:
    α = pm.Normal('α',mu=0,sigma=1.5,shape=np.unique(gid).size)
    p = pm.Deterministic('p',pm.math.invlogit(α[gid]))
    A = pm.Binomial('A',n=d_ucb.applications,p=p,observed=d_ucb.admit)
    
    # compute contrast in the model
    diff_α = pm.Deterministic('diff_α',α[0]-α[1])
    diff_p = pm.Deterministic('diff_p',p[0]-p[1])

In [ ]:
trace_m11_7 = pm.sample(model=m11_7)

In [ ]:
pm.summary(trace_m11_7,var_names=['α','diff_α','diff_p'], round_to=2)

* We observe a positive log-odds, indicating higher probability of admission for males

* On the probability scale, male admissions seem to be favorable for males in: 14% ± 2%

#### Posterior predictive check

* We look at the proportion of admitted candidates (accepted/total) across departments and genders

In [ ]:
ppc = pm.sample_posterior_predictive(trace_m11_7,model=m11_7,var_names=['A']).posterior_predictive['A']
pp_admit = ppc/d_ucb.applications.values

In [ ]:
# adapted from (https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb)
for i in range(6):
    x = 1 + 2 * i

    y1 = d_ucb.admit[x] / d_ucb.applications[x]
    y2 = d_ucb.admit[x + 1] / d_ucb.applications[x + 1]

    plt.plot([x, x + 1], [y1, y2], "-C0o", alpha=0.6, lw=2)
    plt.text(x + 0.25, (y1 + y2) / 2 + 0.05, d_ucb.dept[x])

plt.plot(range(1, 13), [trace_m11_7.posterior["p"][:,:,i].mean().values for i in range(0,d_ucb.admit.size)], 
         "ko", fillstyle="none", ms=6, alpha=0.6)
plt.plot([range(1, 13), range(1, 13)], az.hdi(trace_m11_7.posterior["p"].values).T, "k-", lw=1, alpha=0.6)
plt.plot([range(1, 13), range(1, 13)], az.hdi(pp_admit)['A'].T, "k+", ms=6, alpha=0.6)

plt.xlabel("case")
plt.xticks(range(1,13))
plt.ylabel("admit")
plt.title("Posterior validation check")
plt.ylim(0, 1);

* **[QUESTION]: Are results of the posterior predictive check good? Why?**

#### Model 2


* In what follows, we study the use of another parameter for department

\begin{align*}
A_i \sim & \; \mathrm{Binomial}(N_i,p_i) \\
\mathrm{logit}(p) = & \; \alpha_{\mathrm{GID}[i]} + \delta_{\mathrm{DEPT}[i]} \\
\sigma_j \sim & \; \mathrm{Normal}(0,1.5) \\
\delta_k \sim & \; \mathrm{Normal}(0,1.5) \\
\end{align*}

In [ ]:
dept_id = pd.Categorical(d_ucb["dept"]).codes # get numerical indexes for department
with pm.Model() as m11_8:
    δ = pm.Normal('δ',mu=0,sigma=1.5,shape=np.unique(dept_id).size)
    α = pm.Normal('α',mu=0,sigma=1.5,shape=np.unique(gid).size)
    p = pm.Deterministic('p',pm.math.invlogit(α[gid] + δ[dept_id]))
    A = pm.Binomial('A',n=d_ucb.applications,p=p,observed=d_ucb.admit)
    
    # compute contrast in the model
    diff_p = pm.Deterministic('diff_p',p[0]-p[1])

In [ ]:
trace_m11_8 = pm.sample(model=m11_8)

In [ ]:
pm.summary(trace_m11_8,var_names=['α','δ','diff_p'],round_to=2)

* After adding the department predictor, on the probability scale, males have a slight disadvantage
    * 2% less admission rate on average

<br>

* To understand why, we look at the application rates for the different departments

In [ ]:
pg = pd.DataFrame(index=["male", "female"], columns=d_ucb.dept.unique())
for dep in pg.columns:
    pg[dep] = (
        d_ucb.loc[d_ucb.dept == dep, "applications"]
        / d_ucb.loc[d_ucb.dept == dep, "applications"].sum()
    ).values
pg.round(2)

* The table above shows that departments A and B have the highest male applications, and C and E have more female applicants

<br>

* A quick look at the δ parameters in the summary of `m11_8` also shows that departments A and B have the highest admission rates, whereas C,D and E have the lowest
    * That is, women submitted more applications to departments with low admission rates
    
<br>

* The posterior predictive check (similar to that of `m11_7`) shows that this model better fits the data

In [ ]:
ppc_m11_8 = pm.sample_posterior_predictive(trace_m11_8,model=m11_8,var_names=['A']).posterior_predictive['A']
pp_admit_m11_8 = ppc_m11_8/d_ucb.applications.values

# adapted from (https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb)
for i in range(6):
    x = 1 + 2 * i

    y1 = d_ucb.admit[x] / d_ucb.applications[x]
    y2 = d_ucb.admit[x + 1] / d_ucb.applications[x + 1]

    plt.plot([x, x + 1], [y1, y2], "-C0o", alpha=0.6, lw=2)
    plt.text(x + 0.25, (y1 + y2) / 2 + 0.05, d_ucb.dept[x])

plt.plot(range(1, 13), [trace_m11_8.posterior["p"][:,:,i].mean().values for i in range(0,d_ucb.admit.size)], 
         "ko", fillstyle="none", ms=6, alpha=0.6)
plt.plot([range(1, 13), range(1, 13)], az.hdi(trace_m11_8.posterior["p"].values).T, "k-", lw=1, alpha=0.6)
plt.plot([range(1, 13), range(1, 13)], az.hdi(pp_admit_m11_8)['A'].T, "k+", ms=6, alpha=0.6)

plt.xlabel("case")
plt.xticks(range(1,13))
plt.ylabel("admit")
plt.title("Posterior validation check")
plt.ylim(0, 1);

#### Studying the causal relationships

* Department might be seen as a confound as it might mislead us about the direct causal effect of gender.

* However, department will likely have a genuine effect on admissions.

<img src="figs/admissions-dag.png" width="20%">

* There are two causal paths from gender to admissions: $G \to D \to A$ and $G \to A$.

* By conditioning on $D$ we can check whether the direct causal path $G \to A$ exists
    
    * This is exactly what model `m11_8` tests
    
    * However, be cautious when drawing conclusions, external variables not present in the data may be also affecting the results

* Finally, note that this model is overly parameterized
 
    * The pair plot shows strong correlation between pairs of sampled coefficient values
     
    * Over-parametrization can result in
 
        * Poor performance MCMC inference
          
        * Excessive variance in posterior distributions

In [ ]:
pm.plot_pair(trace_m11_8,var_names=['α','δ']);

## Poisson regression

* The Poisson distribution is used when the outcome variable is a count with an unknown upper limit
    * In the models above, we knew:
        * The total number of experiments for each chimpanzee
        * The total number of applicants per department
    * Consider a model where the theoretical maximum is unknown (it could be unbounded)
        * Number of fish captured in a day
        * Number of drinks at Scrollbar on Friday
        * $\ldots$

### Poisson distribution

* Before we continue, lets try to understand the intuition behind the Poisson distribution
    * Concretely, as a special case of a Binomial distribution with an unbounded number of trials ($n$)

* The pmf of the Poisson distribution is defined as:

$$
p(k \mid \lambda) = \frac{\lambda^k e^{-\lambda}}{k!}
$$

* Let us compute the limit of the Binomial pmf when $n \to \infty$, let $\lambda = p \cdot n$

$$
\begin{align}
P(k) &= \binom{n}{k}\left(\frac{\lambda}{n}\right)^k\left(1- \frac{\lambda}{n}\right)^{n-k}\\
&= \frac{n(n-1)(n-2)\ldots(n-k+1)}{k!}\left(\frac{\lambda}{n}\right)^k\left(1- \frac{\lambda}{n}\right)^{n-k}\\
&= \frac{n(n-1)(n-2)\ldots(n-k+1)}{k!}\frac{\lambda^k}{n^k}\left(1- \frac{\lambda}{n}\right)^{n-k}\\
&= \frac{n}{n}\frac{n-1}{n}\frac{n-2}{n}\frac{n-k+1}{n}\frac{\lambda^k}{k!}\left(1- \frac{\lambda}{n}\right)^{n-k}\\
&= \frac{n}{n}\frac{n-1}{n}\frac{n-2}{n}\frac{n-k+1}{n}\frac{\lambda^k}{k!}\left(1- \frac{\lambda}{n}\right)^{n}\left(1- \frac{\lambda}{n}\right)^{-k}
\end{align}
$$

* Then, $\lim_{n \to \infty} P(k) = \frac{\lambda^k e^{-\lambda}}{k!}$, which equals the Poisson pmf

<br>

* The parameter $\lambda$ in the Poisson distribution is often refer to as the *rate*
    * This parameter corresponds to the expectation and variance of the distribution
    
<br>

* It is important to note that $\lambda$ must be positive, $\lambda \in (0, \infty)$

### Poisson in the GLM

* When the outcome we want to predict ($y$) is an unbounded count, then it is appropriate to use a Poisson  as a data distribution

$$
y \sim \mathrm{Poisson}(\lambda)
$$


* The parameter $\lambda$ models the expected value of the count we are evaluating


* It is necessary to ensure that $\lambda$ is positive
    * To this end, it is common to use a $\mathrm{log}$ link function
    
    \begin{align*}
    &\\
    y_i \sim & \; \mathrm{Poisson}(\lambda_i) \\
    \mathrm{log}(\lambda_i) = & \; \alpha + \beta(x - \bar{x})
    &\\
    &\\
    \end{align*}
    
* The $\mathrm{log}$ link function implies and exponential relation with predictors
    * In nature, there exist phenomena for which exponential does not remain for long

### Oceanic Islands and Tools

<img src="figs/oceanic-islands.png" width="50%">

In [ ]:
d_k = pd.read_csv(os.path.join("Data", "Kline.csv"), delimiter=";")
d_k

#### Geocentric Model

* We want to predict the number of tools based on the population size in the different islands

<br>

* We use a Poisson model because the number of tools does not have a fix upper-bound

<br>

* We would like a model that encapsulates the following assumptions:
    1. The number of tools increases with the $\mathrm{log}$ of population size.
    2. The number of tools increases with the level of contact of the islands with other islands
    3. The impact of $\mathrm{log}$ population on total tools is moderated by "high" contact


In [ ]:
d_k['P'] = (np.log(d_k.population) - np.log(d_k.population).mean()) / np.log(d_k.population).std()
c_id = (d_k.contact == "high").astype(int).values

* Formally the model is defined as

\begin{align*}
T \sim & \; \mathrm{Poisson}(\lambda_i) \\
\mathrm{log}(\lambda_i) = & \; \alpha_{\mathrm{CID[i]}} + \beta_{\mathrm{CID}[i]}\mathrm{log}P_i\\
\beta_j \sim & \; \text{To be determined} \\
\alpha_j \sim & \; \text{To be determined} \\
\end{align*}

##### Finding a prior for $\alpha$

* As for the Binomial model above, flat priors in the linear model are not flat in the outcome scale

<br>

* In what follows, we explore some possible parameters for the priors
    * First we try a flat prior (in the linear model)

<br>

* We sample directly from a LogNormal distribution    
    * This is the same as sampling from a Normal distribution and compute the power of $e$, i.e., $X \sim \mathrm{Normal}(0,1)$, $\alpha = e^X$    
    * Recall that if you compute the log a of LogNormal random variable you transform it into a Normal random variable

In [ ]:
az.plot_posterior(
    # np.exp(pm.draw(pm.Normal.dist(0, 10), draws=20_000)),
    pm.draw(pm.LogNormal.dist(0, 10),draws=20_000),
    label="α ~ LogNormal(0, 10)"
)
plt.show();

* Note the extremely large mean which is unrealistic 
    * Obviously, there will be no islands with that number of tools

<br>

* Below we plot a more adequate prior

In [ ]:
az.plot_posterior(
    # np.exp(pm.draw(pm.Normal.dist(3, 0.5), draws=20_000)),
    pm.draw(pm.LogNormal.dist(3, 0.5), draws=20_000),
    label="α ~ LogNormal(3, 0.5)"
)
plt.title('')
plt.show();

##### Finding a prior for $\beta$

* Now the prior for $\alpha$ as $\mathrm{Normal}(3,0.5)$ is fixed.

* In what follows we explore different priors for $\beta$

In [ ]:
# copied from (https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb)

def kline_prior_plot(N: int = 100, b_prior: str = "bespoke", x_scale: str = "stdz", ax=None):
    """
    Utility function to plot prior predictive checks for Kline Poisson model.
    N: number of prior predictive trends.

    """
    if ax is None:
        _, ax = plt.subplots()
    ax.set_ylabel("total tools")

    itcpts = np.random.normal(3.0, 0.5, N)
    if b_prior == "conventional":
        slopes = np.random.normal(0.0, 10.0, N)
        ax.set_title("b ~ Normal(0, 10)")
    elif b_prior == "bespoke":
        slopes = np.random.normal(0.0, 0.2, N)
        ax.set_title("b ~ Normal(0, 0.2)")
    else:
        raise ValueError(
            "Prior for slopes (b_prior) can only be either 'conventional' or 'bespoke'."
        )

    x_seq = np.linspace(np.log(100), np.log(200_000), N)
    ax.set_ylim((0, 500))
    if x_scale == "log":
        for a, b in zip(itcpts, slopes):
            ax.plot(x_seq, np.exp(a + b * x_seq), "k", alpha=0.4)
        ax.set_xlabel("log population")
    elif x_scale == "natural":
        for a, b in zip(itcpts, slopes):
            ax.plot(np.exp(x_seq), np.exp(a + b * x_seq), "k", alpha=0.4)
        ax.set_xlabel("population")
    else:
        x_seq = np.linspace(-2, 2, N)
        for a, b in zip(itcpts, slopes):
            ax.plot(x_seq, np.exp(a + b * x_seq), "k", alpha=0.4)
        ax.set_ylim((0, 100))
        ax.set_xlabel("log population (std)")

    return ax

In [ ]:
# copied from (https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb)
_, ax = plt.subplots(2, 2, figsize=(10, 10))
kline_prior_plot(b_prior="conventional", x_scale="stdz", ax=ax[0][0])
kline_prior_plot(b_prior="bespoke", x_scale="stdz", ax=ax[0][1])
kline_prior_plot(b_prior="bespoke", x_scale="log", ax=ax[1][0])
kline_prior_plot(x_scale="natural", ax=ax[1][1])
plt.tight_layout();

* Note that the top-left prior models either exponential decay or growth of total number of tools very close to the mean
    * **[QUESTION]: Is the top-left prior a good prior? Why?**

<br>
    
* The second prior (top-right) is more flat for this case of a $\mathrm{log}$ predictor, which gives the data higher weight on the posterior

<br>

* The plots at the bottom show the same prior as top-right but without standarizing the population (bottom-left) and without applying the $\log$ (bottom-right)
    * These are better ways to inspect the prior, as population has a natural zero point
    * Standarized data is mostly useful for inference



##### Model comparison and posterior predictive check

In [ ]:
with pm.Model() as m11_9:
    α = pm.Normal('α',mu=3,sigma=.5)
    λ = pm.Deterministic('λ',pm.math.exp(α))
    T = pm.Poisson('T',mu=λ,observed=d_k.total_tools)
    
with pm.Model() as m11_10:
    α = pm.Normal('α',mu=3,sigma=.5,shape=np.unique(c_id).size)
    β = pm.Normal('β',mu=0,sigma=.2,shape=np.unique(c_id).size)
    
    cid = pm.Data("cid", c_id)
    P_ = pm.Data("P", d_k.P)
    λ = pm.Deterministic('λ',pm.math.exp(α[cid] + β[cid]*P_))
    T = pm.Poisson('T',mu=λ,observed=d_k.total_tools)

In [ ]:
trace_m11_9 = pm.sample(model=m11_9,idata_kwargs={'log_likelihood': True})
trace_m11_10 = pm.sample(model=m11_10,idata_kwargs={'log_likelihood': True})

In [ ]:
pm.compare({'m11_9':trace_m11_9, 'm11_10': trace_m11_10 })

* Note the warning on the large value of $k$ in the Pareto distribution
    * This indicates that there are data points with high influence
<br>

* Additionally, note the larger `p_loo` value for the model only with the intercept
    * This break our intuition that models with more parameters fit better the data
    * The reason is that this intuition only holds for purely linear models

<br>
 
* Let's have a look at the posterior predictive check to verify these points

In [ ]:
# store pareto-k values for plot:
k = az.loo(trace_m11_10, pointwise=True).pareto_k.values

In [ ]:
ns = 10
P_seq = np.linspace(-1.4, 3.0, ns)

with m11_10:
    # predictions for cid=0 (low contact)
    pm.set_data({"cid": np.array([0] * ns), "P": P_seq})
    lam0 = pm.sample_posterior_predictive(trace_m11_10, var_names=["T"])['posterior_predictive']["T"]

    # predictions for cid=1 (high contact)
    pm.set_data({"cid": np.array([1] * ns)})
    lam1 = pm.sample_posterior_predictive(trace_m11_10, var_names=["T"])['posterior_predictive']["T"]

lmu0, lmu1 = lam0.mean(["chain", "draw"]), lam1.mean(["chain", "draw"])

In [ ]:
# adapted from https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb
_, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 6))

# scale point size to Pareto-k:
k /= k.max()
psize = 250 * k

# Plot on standardized log scale:

az.plot_hdi(P_seq, lam1, color="b", fill_kwargs={"alpha": 0.2}, ax=ax0)
ax0.plot(P_seq, lmu1, color="b", alpha=0.7, label="high contact mean")

az.plot_hdi(P_seq, lam0, color="k", fill_kwargs={"alpha": 0.2}, ax=ax0)
ax0.plot(P_seq, lmu0, "--", color="k", alpha=0.7, label="low contact mean")

# display names and k:
mask = k > 0.3
labels = d_k.culture.values[mask]
for i, text in enumerate(labels):
    ax0.text(
        d_k.P.values[mask][i] - 0.2,
        d_k.total_tools.values[mask][i] + 4,
        f"{text} ({np.round(k[mask][i], 2)})",
        fontsize=8,
    )

# display observed data:
index = c_id == 1
ax0.scatter(
    d_k.P[~index],
    d_k.total_tools[~index],
    s=psize[~index],
    facecolors="none",
    edgecolors="k",
    alpha=0.8,
    lw=1,
    label="low contact",
)
ax0.scatter(d_k.P[index], d_k.total_tools[index], s=psize[index], alpha=0.8, label="high contact")
ax0.set_xlabel("log population (std)")
ax0.set_ylabel("total tools")
ax0.legend(fontsize=8, ncol=2)

# Plot on natural scale:
# unstandardize and exponentiate values of standardized log pop:
P_seq = np.linspace(-5.0, 3.0, ns)
P_seq = np.exp(P_seq * np.log(d_k.population.values).std() + np.log(d_k.population.values).mean())

az.plot_hdi(P_seq, lam1, color="b", fill_kwargs={"alpha": 0.2}, ax=ax1)
ax1.plot(P_seq, lmu1, color="b", alpha=0.7)

az.plot_hdi(P_seq, lam0, color="k", fill_kwargs={"alpha": 0.2}, ax=ax1)
ax1.plot(P_seq, lmu0, "--", color="k", alpha=0.7)

# display observed data:
ax1.scatter(
    d_k.population[~index],
    d_k.total_tools[~index],
    s=psize[~index],
    facecolors="none",
    edgecolors="k",
    alpha=0.8,
    lw=1,
)
ax1.scatter(d_k.population[index], d_k.total_tools[index], s=psize[index], alpha=0.8)
plt.setp(ax1.get_xticklabels(), ha="right", rotation=45)
ax1.set_xlim((-10_000, 350_000))
ax1.set_xlabel("population")
ax1.set_ylabel("total tools");

* It is noticeable the $k$ value of Hawaii (1.01) due to its large population size (compared to the other island)

* It is important to note that we should not discard influencial data points (like Hawaii), they simply have a strong impact on the shape of the posterior

* The plot above also shows a lot of uncertainty for the islands with high contact

#### Scientific model

* The trend above gives higher total tools count for low contact islands after certain population size
    * This is unintuitive as high contact Hawaii should have as many tools as the existing Hawaii

<br>
    
* The reason for the above is that the intercept allows for total tool counts larger than zero for islands with zero population size!

$$
\log(\lambda_i) = \alpha_{\mathrm{CID[i]}} + \beta_{\mathrm{CID}[i]}\log(P_i)
$$

* To address this issue, the book proposes a *scientific model*, i.e., a model whose shape has some meaning within its domain of application

* The model states that the change on tool count, $\Delta T$, is related to the population size, $P$ and total tools count $T$ as follows

$$
\Delta T = \alpha P^\beta - \gamma T
$$

* Solving it for $T$ with $\Delta T = 0$ gives us the rate (namely, $\lambda$) of total tool count
$$
\lambda = \hat{T} = \frac{\alpha P^\beta}{\gamma}
$$

* This results in the following GLM

\begin{align*}
T_i \sim & \; \mathrm{Poisson}(\lambda_i)\\
\lambda_i = & \; \frac{\alpha P_i^\beta}{\gamma}\\
&\text{some priors for } \alpha, \beta, \gamma
\end{align*}

##### Posterior predictive check

* The posterior predictive check on this model looks much more reasonable

In [ ]:
with pm.Model() as m11_11:
        γ   = pm.Exponential('γ',1)
        β   = pm.Exponential('β',1,shape=np.unique(c_id).size)
        α   = pm.LogitNormal('α',1,1,shape=np.unique(c_id).size)
        cid = pm.Data("cid", c_id)
        P   = pm.Data("P", d_k.population)
        λ   = pm.Deterministic('λ', α[cid]*tt.math.pow(P,β[cid])/γ)
        T   = pm.Poisson('T',mu=λ,observed=d_k.total_tools)
        trace_m11_11 = pm.sample(idata_kwargs={'log_likelihood': True})

In [ ]:
# Adapted from https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb
ns = 10
P_seq = d_k.population

with m11_11:
    # predictions for cid=0 (low contact)
    pm.set_data({"cid": np.array([0] * ns), "P": P_seq})
    lam0 = pm.sample_posterior_predictive(trace_m11_11, var_names=["T"])[
        "posterior_predictive"
    ]["T"]

    # predictions for cid=1 (high contact)
    pm.set_data({"cid": np.array([1] * ns)})
    lam1 = pm.sample_posterior_predictive(trace_m11_11, var_names=["T"])[
        "posterior_predictive"
    ]["T"]

lmu0, lmu1 = lam0.mean(["chain", "draw"]), lam1.mean(["chain", "draw"])

# store pareto-k values for plot:
k = az.loo(trace_m11_11, pointwise=True).pareto_k.values

In [ ]:
# Adapted from https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynbb
_,  ax1 = plt.subplots(1, figsize=(7, 6))

# scale point size to Pareto-k:
k /= k.max()
psize = 250 * k

az.plot_hdi(P_seq, lam1, color="b", fill_kwargs={"alpha": 0.2}, ax=ax1)
ax1.plot(P_seq, lmu1, color="b", alpha=0.7, label="high contact mean")

az.plot_hdi(P_seq, lam0, color="k", fill_kwargs={"alpha": 0.2}, ax=ax1)
ax1.plot(P_seq, lmu0, "--", color="k", alpha=0.7, label="low contact mean")

# display observed data:
ax1.scatter(
    d_k.population[~index],
    d_k.total_tools[~index],
    s=psize[~index],
    facecolors="none",
    edgecolors="k",
    alpha=0.8,
    lw=1,
    label='low contact'
)
ax1.scatter(d_k.population[index], d_k.total_tools[index], s=psize[index], alpha=0.8, label='high contact')
plt.setp(ax1.get_xticklabels(), ha="right", rotation=45)
ax1.set_xlim((-10_000, max(P_seq)))
ax1.set_xlabel("population")
ax1.legend(fontsize=8, ncol=2)
ax1.set_ylabel("total tools");

##### Model comparison (with previous two models)

In [ ]:
pm.compare({'m11_9':trace_m11_9, 'm11_10': trace_m11_10, 'm11_11': trace_m11_11})

## Multinomial Regression

* The Binomial distribution is useful only when there are two possible outcomes
    * Either the chimpanzee pulls left or not
    * Either a student is admitted or not
    * Taking either a blue or white marble
    
<br>


* The Multinomial distribution is a generalization of the Binomial distribution to account for any number of possible (unordered) outcomes
    * Taking either a blue or white **or red** marble

<br>

* As the Binomial, the Multinomial distribution is a maxent distribution


<br>


* The pmf of the Multinomial distribution is

$$
p(y_1, \ldots, y_k \mid n, p_1, \ldots, p_k) = \frac{n!}{\Pi_i y_i!} \Pi^k_{i=1}p^{y_i}_i
$$

* With $k=2$ and $n=1$, Multinomial equals Bernoulli
    * Two outcomes $\{0,1\}$ and $n=1$ trial

<br>


* With $k=2$ and $n>1$, Multinomial equals Binomial
    * Two outcomes $\{0,1\}$ and $n>1$ trials

<br>

* When $n=1$ the Multinomial distribution is also known as **Categorical**

<br>

* Examples
    * Consider again the marbles example,  with 3 white marbles, 4 blue marbles and 1 red marble
    * Imagine that we take 3 marbles, what is the probability of taking one of each color?

In [ ]:
p = [
    3/8, # 3 white marbles
    4/8, # 4 blue marbles
    1/8  # 1 red marble
         # 8 marbles in total
]
multinomial(n=3,p=p).pmf([1,1,1])

### Multinomial GLM

* The standard (inverse) link function for a multinomial GLM is the multinomial logit or **softmax**
    * It simply takes a vector of scores for each outcome and turns them into a probability
    $$
    p(k \mid s_1, \ldots, s_K) = \frac{\exp(s_k)}{\sum^K_{i=1}\exp(s_i)}
    $$
    * A model using this inverse link function is known as **multinomial logistic regression**

## Example on simulated data: career based on income

### Predictor values depend on outcome

* We try to predict career choice based on the expected income for that career
    * The outcome *career* is directly related to the predictor *income* (per career)

<br>

* The code below simulates the career choices of 500 individuals given some income data

In [ ]:
# copied from (https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb)

# simulate career choices among 500 individuals
N = 500  # number of individuals
income = np.array([1, 2, 5])  # expected income of each career
score = 0.5 * income  # score for each career, based on income
# converts scores to probabilities:
p = softmax(score)

# now simulate choice
# outcome career holds event type values, not counts
career = np.random.multinomial(1, p, size=N)
career = np.where(career == 1)[1]
career[:11], score, p

* Below we use a slightly modified model taken from the PyMC repository

In [ ]:
# copied from (https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb)
with pm.Model() as m11_13:
    a = pm.Normal("a", 0.0, 1.0, shape=2)  # intercepts
    b = pm.Normal("b", 0.0, 0.5)  # association of income with choice

    s0 = a[0] + b * income[0]
    s1 = a[1] + b * income[1]
    s2 = 0.0  + b * income[2]  # pivoting the intercept for the third category
    s = pm.math.stack([s0, s1, s2])

    p_ = pm.math.softmax(s)
    career_obs = pm.Categorical("career", p=p_, observed=career)

    trace_11_13 = pm.sample(tune=2000, target_accept=0.99)
az.summary(trace_11_13, round_to=2)

* The most important thing to remember is that these results are w.r.t. the chosen pivot (in this case `s2`)
* However, the posterior of parameter `b` shows a clear positive tendency
    * We can conclude that expected income has positive effect on career choice
        * As expected, as we set it to 0.5 when generating the data
    * Based on the value of `b` in this scale, we cannot determine the absolute increase in probability

#### Counterfactual analysis

* We can use the posterior distribution to compute the increase of probability in an artificially introduced career with higher income

* Concretely we recompute the scores for career 1 with double income and then compute the difference

In [ ]:
# copied from (https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb)

# set up logit scores:
post_11_13 = az.extract(trace_11_13["posterior"])
s0 = post_11_13["a"][0, :] + post_11_13["b"] * income[0]
s1_orig = post_11_13["a"][1, :] + post_11_13["b"] * income[1]
s1_new = post_11_13["a"][1, :] + post_11_13["b"] * income[1] * 2
s2 = 0.0 + post_11_13["b"] * income[2]

pp_scores_orig = np.stack([s0, s1_orig, s2]).T
pp_scores_new = np.stack([s0, s1_new, s2]).T

# compute probabilities for original and counterfactual:
p_orig = softmax(pp_scores_orig, axis=1)
p_new = softmax(pp_scores_new, axis=1)

# summarize
p_diff = p_new[:, 1] - p_orig[:, 1]
az.summary({"p_diff": p_diff}, kind="stats", round_to=2)

* The results indicate an expected increase of ~19% in probability of choosing a career when the income is doubled.

    * Note that this result is conditional on comparing to other career options

### Predictor values fixed for all outcomes

* The model above matches predictors to outcome
    * The income is per career choice

<br>


* Alternatively, we can match the outcome to a predictor value, e.g., family income
    * Now there are as many family incomes as records and we use them to estimate their effect on career choice
    * The outcome *career* is related to the predictor *family income* (per subject, not per career as above)
    
<br>
    
    
* As before, we simulate the data

In [ ]:
# adapted from (https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb)
N = 500

# simulate family incomes for each individual
family_income = np.random.rand(N)
print(family_income[:11])

# assign a unique coefficient for each type of event
b = np.array([-2.0, 0.0, 2.0])

p = softmax(np.array([0.5, 1.0, 1.5])[:, None] + np.outer(b, family_income), axis=0).T

career = np.asarray([np.random.multinomial(1, pp) for pp in p])
career = np.where(career == 1)[1]
career[:11]

* Now the scores in the model are multiplied by each family income
    * As opposed to before where they were multiplied by the expected income of each career path

In [ ]:
# adapted from (https://github.com/pymc-devs/pymc-resources/blob/main/Rethinking_2/Chp_11.ipynb)

with pm.Model() as m11_14:
    a = pm.Normal("a", 0.0, 1.5, shape=2)  # intercepts
    b = pm.Normal("b", 0.0, 1, shape=2)  # coefficients on family income

    fi = pm.Data('family_income', family_income)

    s0 = a[0] + b[0] * family_income
    s1 = a[1] + b[1] * family_income
    s2 = pm.math.zeros(N)  # pivot
    s = pm.math.stack([s0, s1, s2]).T

    p_ = pm.Deterministic('p',pm.math.softmax(s, axis=1))
    career_obs = pm.Categorical("career", p=p_, observed=career, shape=fi.shape)

    trace_11_14 = pm.sample(1000, tune=2000, target_accept=0.9)
pm.summary(trace_11_14, round_to=2)

* These values are to be interpreted with respect to the pivot `s2`
    * For instance, `b[2] == 2.0`, which explains why `b[1]` shows negative value for the entire HDI

<br>

* As before, we can plot posterior predictions to compare the effect of different family income values

In [ ]:
incomes = np.linspace(family_income.min(), family_income.max(), num=3)
pm.set_data({'family_income': np.array(incomes)}, model=m11_14)
pp_11_14 = pm.sample_posterior_predictive(trace_11_14, model=m11_14, predictions=True, var_names=['p'])

In [ ]:
pm.plot_forest(
    [pp_11_14.predictions.sel(p_dim_0=i) for i in range(len(incomes))],
    model_names=[f'Income {np.round(i,3)}' for i in incomes],
    combined=True);